In [3]:
import json
import sys
sys.path.append(r"/Users/alejandrogomez-paz/Desktop/RAG/local_LLM")
from rad_ai import query   # Ollama-served rad-ai model

CORPUS_PATH = r"/Users/alejandrogomez-paz/Desktop/RAG/corpus_v1/corpus_1.jsonl"

chunks, metas = [], []
with open(CORPUS_PATH, encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)
        if r.get("record_type") != "chunk":
            continue  # skip the meta record
        chunks.append(r["text"])
        metas.append({
            "source": r["source"],
            "page": r["section"],        # keeps rag_answer's meta['page'] working unchanged
            "chunk_id": r["chunk_id"],   # what golden_pairs_1 matches on
        })

print("Total chunks:", len(chunks))
print("Example meta:", metas[0])

import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

emb = embedder.encode(chunks, normalize_embeddings=True, show_progress_bar=True)
emb = np.asarray(emb, dtype=np.float32)

dim = emb.shape[1]
index = faiss.IndexFlatIP(dim)   # cosine similarity if embeddings normalized
index.add(emb)

print("FAISS index size:", index.ntotal)

# =============== (was Cell 3) HF login ===============
# CHANGED: made login optional. Only needed for gated models (e.g. Mistral/Gemma).
# Uncomment and run once, OR set HUGGINGFACE_HUB_TOKEN in your env, OR use an open model below.
# from huggingface_hub import login
# login()


# =============== (Cell 4) load LLM — STABLE CPU PATH ===============
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# =============== (Cell 5) retrieve + rag_answer — UNCHANGED ===============
def retrieve(question: str, k: int = 5):
    qv = embedder.encode([question], normalize_embeddings=True).astype(np.float32)
    scores, ids = index.search(qv, k)
    results = []
    for score, idx in zip(scores[0], ids[0]):
        results.append((float(score), chunks[idx], metas[idx]))
    return results

def rag_answer(question: str, k: int = 5, max_new_tokens: int = 300):
    results = retrieve(question, k=k)

    context_blocks = []
    for score, txt, meta in results:
        context_blocks.append(
            f"[SOURCE: {meta['source']} | page {meta['page']} | score {score:.3f}]\n{txt}"
        )
    context = "\n\n---\n\n".join(context_blocks)

    system = (
        "You are RAD-AI, a radiation safety assistant.\n"
        "Use ONLY the provided context to answer.\n"
        "If the answer is not in the context, say: "
        "'I don't have enough information in the provided files.'\n"
        "Cite sources as (source, page)."
    )

    user = f"CONTEXT:\n{context}\n\nQUESTION:\n{question}"

    return query(user, system=system, temperature=0.0)

Total chunks: 41
Example meta: {'source': 'OSHA — Radiation Emergency Preparedness and Response', 'page': 'Health and Safety Hazards during Radiation Emergencies', 'chunk_id': 'osha_rep_hazards'}


Batches: 100%|██████████| 2/2 [00:00<00:00,  7.76it/s]

FAISS index size: 41


In [3]:
print(rag_answer("What is the dose threshold for acute radiation syndrome?", k=5))

user
You are RAD-AI, a radiation safety assistant.
Use ONLY the provided context to answer.
If the answer is not in the context, say: 'I don't have enough information in the provided files.'
Cite sources as (source, page).

CONTEXT:
[SOURCE: afrri_med_mgmt_handbook.pdf | page 4 | score 0.701]
For externally irradiated patients without trauma, patients receiving a high dose can be distinguished from those with a dose < 1 Gy
using two criteria: the neutrophil/lymphocyte (N/L) ratio and whether emesis has occurred. A triage score T is assigned as follows.
T = N/L + E, where E = 0 if no emesis; E = 2 if emesis.
In a normal, healthy human population, the N/L ratio from a complete blood count (CBC) with differential has been found to be ~ 2.1.
For time > 4 h post-event, T is significantly elevated for doses > 1 Gy. One study has shown this scoring technique to have a
sensitivity of 89% and a specificity of 93% to separate those with doses < 1 Gy from those with higher doses (N = 226 controls